# PINNsFormer reproducibility study

Reproduces Table 1 (and parts of Table 2) of *PINNsFormer: A Transformer-Based Framework for Physics-Informed Neural Networks* (Zhao, Ding, Prakash, ICLR 2024) plus a few experiments beyond the paper.

**Runtime -> Change runtime type -> GPU (T4 is enough).** Then run the cells top to bottom. Results accumulate in `pinnsformer/results/results.csv`; the last cells copy them to Google Drive so nothing is lost when the session dies.

In [ ]:
!nvidia-smi
!git clone -q https://github.com/AdityaLab/pinnsformer.git
%cd pinnsformer
!pip -q install scipy

## Experiment code
The three cells below write our runner into the cloned repo.

In [ ]:
%%writefile run_experiment.py
"""
Unified experiment runner for the PINNsFormer reproducibility study.

Re-implements the training/evaluation loop of the authors' demo notebooks
(demo/convection, demo/1d_reaction, demo/1d_wave) as one script so that every
model / PDE / seed combination is run identically and logged to a CSV.

Examples
--------
  # Reproduce Table 1, convection, PINNsFormer, exactly as in the notebook
  python run_experiment.py --pde convection --model pinnsformer

  # Plain PINN baseline
  python run_experiment.py --pde convection --model pinn

  # Beyond the paper: PINN with Wavelet activation instead of Tanh
  python run_experiment.py --pde convection --model pinn_wavelet

  # Beyond the paper: PINNsFormer with k=1 (no pseudo-sequence)
  python run_experiment.py --pde 1d_reaction --model pinnsformer --k 1

  # Fairness: PINN on the same 51x51 training grid PINNsFormer uses
  python run_experiment.py --pde convection --model pinn --train_grid 51

Every run appends one row to results/results.csv and writes
results/<run_name>/{pred.npy,loss.json,figure.png,config.json}.
"""

import argparse
import csv
import json
import os
import random
import sys
import time

import numpy as np
import torch
import torch.nn as nn
from torch.optim import LBFGS

ROOT = os.path.dirname(os.path.abspath(__file__))
sys.path.insert(0, ROOT)

from util import get_data, make_time_sequence, get_n_params  # noqa: E402
from model.pinn import PINNs  # noqa: E402
from model.qres import QRes  # noqa: E402
from model.fls import FLS  # noqa: E402
from model.pinnsformer import PINNsformer, WaveAct  # noqa: E402


# --------------------------------------------------------------------------- #
# PDE definitions (mirroring Appendix B of the paper and the demo notebooks)
# --------------------------------------------------------------------------- #

class PDE:
    """Holds domain, residual, IC/BC losses and ground truth for one PDE."""

    name = ""
    x_range = (0.0, 1.0)
    t_range = (0.0, 1.0)

    def residual(self, model, x, t):
        raise NotImplementedError

    def ic_bc_loss(self, model, x_left, t_left, x_upper, t_upper, x_lower, t_lower):
        """Returns (loss_ic, loss_bc). 'left' is t=0, 'upper'/'lower' are x=max/min."""
        raise NotImplementedError

    def ground_truth(self, xt):
        """xt: (N,2) numpy array of test points -> (N,) exact solution."""
        raise NotImplementedError


def grad(outputs, inputs):
    return torch.autograd.grad(outputs, inputs, grad_outputs=torch.ones_like(outputs),
                               retain_graph=True, create_graph=True)[0]


class Convection(PDE):
    """u_t + beta u_x = 0, x in [0,2pi], t in [0,1]; IC sin(x); periodic BC; beta=50."""
    name = "convection"
    x_range = (0.0, 2 * np.pi)
    beta = 50.0

    def residual(self, model, x, t):
        u = model(x, t)
        u_x, u_t = grad(u, x), grad(u, t)
        return u_t + self.beta * u_x, u

    def ic_bc_loss(self, model, x_left, t_left, x_upper, t_upper, x_lower, t_lower):
        pred_left = model(x_left, t_left)
        pred_upper = model(x_upper, t_upper)
        pred_lower = model(x_lower, t_lower)
        loss_ic = torch.mean((pred_left[:, 0] - torch.sin(x_left[:, 0])) ** 2)
        loss_bc = torch.mean((pred_upper - pred_lower) ** 2)
        return loss_ic, loss_bc

    def ground_truth(self, xt):
        import scipy.io
        mat = scipy.io.loadmat(os.path.join(ROOT, "demo", "convection", "convection.mat"))
        return mat["u"].reshape(-1)


class Reaction1D(PDE):
    """u_t - rho u (1-u) = 0, x in [0,2pi]; IC gaussian bump; periodic BC; rho=5."""
    name = "1d_reaction"
    x_range = (0.0, 2 * np.pi)
    rho = 5.0

    def residual(self, model, x, t):
        u = model(x, t)
        u_t = grad(u, t)
        return u_t - self.rho * u * (1 - u), u

    def ic_bc_loss(self, model, x_left, t_left, x_upper, t_upper, x_lower, t_lower):
        pred_left = model(x_left, t_left)
        pred_upper = model(x_upper, t_upper)
        pred_lower = model(x_lower, t_lower)
        h = torch.exp(-(x_left[:, 0] - torch.pi) ** 2 / (2 * (torch.pi / 4) ** 2))
        loss_ic = torch.mean((pred_left[:, 0] - h) ** 2)
        loss_bc = torch.mean((pred_upper - pred_lower) ** 2)
        return loss_ic, loss_bc

    def ground_truth(self, xt):
        x, t = xt[:, 0], xt[:, 1]
        h = np.exp(-(x - np.pi) ** 2 / (2 * (np.pi / 4) ** 2))
        return h * np.exp(self.rho * t) / (h * np.exp(self.rho * t) + 1 - h)


class Wave1D(PDE):
    """u_tt - c u_xx = 0 on [0,1]^2. NOTE: paper text says beta=3 but the analytic
    solution and the authors' code both correspond to c = 4."""
    name = "1d_wave"
    x_range = (0.0, 1.0)
    c = 4.0

    def residual(self, model, x, t):
        u = model(x, t)
        u_x = grad(u, x)
        u_xx = grad(u_x, x)
        u_t = grad(u, t)
        u_tt = grad(u_t, t)
        return u_tt - self.c * u_xx, u

    def ic_bc_loss(self, model, x_left, t_left, x_upper, t_upper, x_lower, t_lower):
        pred_left = model(x_left, t_left)
        pred_upper = model(x_upper, t_upper)
        pred_lower = model(x_lower, t_lower)
        pi = torch.pi
        ui_t = grad(pred_left, t_left)
        loss_ic = torch.mean((pred_left[:, 0] - torch.sin(pi * x_left[:, 0])
                              - 0.5 * torch.sin(3 * pi * x_left[:, 0])) ** 2) \
            + torch.mean(ui_t ** 2)
        loss_bc = torch.mean(pred_upper ** 2) + torch.mean(pred_lower ** 2)
        return loss_ic, loss_bc

    def ground_truth(self, xt):
        x, t = xt[:, 0], xt[:, 1]
        return np.sin(np.pi * x) * np.cos(2 * np.pi * t) + 0.5 * np.sin(3 * np.pi * x) * np.cos(6 * np.pi * t)


PDES = {"convection": Convection, "1d_reaction": Reaction1D, "1d_wave": Wave1D}


# --------------------------------------------------------------------------- #
# Models
# --------------------------------------------------------------------------- #

def replace_activation(module, old_cls, new_factory):
    for name, child in module.named_children():
        if isinstance(child, old_cls):
            setattr(module, name, new_factory())
        else:
            replace_activation(child, old_cls, new_factory)


class SinAct(nn.Module):
    def forward(self, x):
        return torch.sin(x)


ACTIVATIONS = {"wavelet": WaveAct, "sin": SinAct, "relu": nn.ReLU, "sigmoid": nn.Sigmoid, "tanh": nn.Tanh}


def build_model(args):
    """Returns (model, is_sequence_model)."""
    if args.model == "pinn":
        return PINNs(in_dim=2, hidden_dim=512, out_dim=1, num_layer=4), False
    if args.model == "pinn_wavelet":
        m = PINNs(in_dim=2, hidden_dim=512, out_dim=1, num_layer=4)
        replace_activation(m, nn.Tanh, WaveAct)
        return m, False
    if args.model == "fls":
        return FLS(in_dim=2, hidden_dim=512, out_dim=1, num_layer=4), False
    if args.model == "qres":
        return QRes(in_dim=2, hidden_dim=args.qres_hidden, out_dim=1, num_layer=args.qres_layers), False
    if args.model == "pinnsformer":
        m = PINNsformer(d_out=1, d_hidden=512, d_model=args.d_model, N=args.n_layers, heads=args.heads)
        if args.activation != "wavelet":  # Table 6 ablation: swap every Wavelet activation
            replace_activation(m, WaveAct, ACTIVATIONS[args.activation])
        return m, True
    raise ValueError(args.model)


def init_weights(m):
    if isinstance(m, nn.Linear):
        torch.nn.init.xavier_uniform_(m.weight)
        m.bias.data.fill_(0.01)


# --------------------------------------------------------------------------- #
# Main
# --------------------------------------------------------------------------- #

def main():
    p = argparse.ArgumentParser(description=__doc__, formatter_class=argparse.RawDescriptionHelpFormatter)
    p.add_argument("--pde", choices=PDES.keys(), required=True)
    p.add_argument("--model", choices=["pinn", "pinn_wavelet", "fls", "qres", "pinnsformer"], required=True)
    p.add_argument("--seed", type=int, default=0)
    p.add_argument("--iters", type=int, default=500,
                   help="L-BFGS outer steps. Authors' notebooks use 500 (paper text says 1000).")
    p.add_argument("--train_grid", type=int, default=None,
                   help="Training mesh size per axis. Default: 51 for pinnsformer, 101 otherwise (as in the notebooks).")
    p.add_argument("--test_grid", type=int, default=101)
    p.add_argument("--k", type=int, default=5, help="pseudo-sequence length (pinnsformer only)")
    p.add_argument("--dt", type=float, default=1e-4,
                   help="pseudo-sequence step. All non-NTK notebooks use 1e-4 (the 1d_wave NTK notebook uses 1e-3).")
    p.add_argument("--activation", choices=ACTIVATIONS.keys(), default="wavelet",
                   help="pinnsformer only: replace the Wavelet activation everywhere (paper Table 6 ablation).")
    p.add_argument("--qres_hidden", type=int, default=256,
                   help="QRes width. Paper Table 4 and the 1d_reaction notebook use 256; the convection notebook uses 512.")
    p.add_argument("--qres_layers", type=int, default=4,
                   help="QRes num_layer. Paper and 1d_reaction notebook use 4; the convection notebook uses 2.")
    p.add_argument("--d_model", type=int, default=32)
    p.add_argument("--heads", type=int, default=2)
    p.add_argument("--n_layers", type=int, default=1)
    p.add_argument("--device", default=None, help="cuda / cpu / mps. Default: cuda if available else cpu.")
    p.add_argument("--out", default=os.path.join(ROOT, "results"))
    p.add_argument("--tag", default="", help="extra string appended to the run name")
    p.add_argument("--save_model", action="store_true")
    args = p.parse_args()

    # ---- defaults that follow the authors' notebooks
    is_seq = args.model == "pinnsformer"
    if args.train_grid is None:
        args.train_grid = 51 if is_seq else 101
    device = args.device or ("cuda" if torch.cuda.is_available() else "cpu")

    # ---- seeding
    np.random.seed(args.seed)
    random.seed(args.seed)
    torch.manual_seed(args.seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(args.seed)

    pde = PDES[args.pde]()
    run_name = f"{args.pde}_{args.model}_g{args.train_grid}_it{args.iters}_s{args.seed}"
    if is_seq:
        run_name += f"_k{args.k}_dt{args.dt:g}"
        if args.activation != "wavelet":
            run_name += f"_{args.activation}"
    if args.tag:
        run_name += f"_{args.tag}"
    run_dir = os.path.join(args.out, run_name)
    os.makedirs(run_dir, exist_ok=True)
    print(f"[run] {run_name}  device={device}")

    # ---- data
    res, b_left, b_right, b_upper, b_lower = get_data(pde.x_range, pde.t_range, args.train_grid, args.train_grid)
    res_test, _, _, _, _ = get_data(pde.x_range, pde.t_range, args.test_grid, args.test_grid)
    n_res, n_ic, n_bc = len(res), len(b_left), len(b_upper)

    if is_seq:
        seq = lambda a: make_time_sequence(a, num_step=args.k, step=args.dt)  # noqa: E731
        res, b_left, b_upper, b_lower = map(seq, (res, b_left, b_upper, b_lower))
        res_test_in = seq(res_test)
    else:
        res_test_in = res_test

    def to_t(a):
        return torch.tensor(a, dtype=torch.float32, requires_grad=True).to(device)

    res, b_left, b_upper, b_lower = map(to_t, (res, b_left, b_upper, b_lower))
    x_res, t_res = res[..., 0:1], res[..., 1:2]
    x_left, t_left = b_left[..., 0:1], b_left[..., 1:2]
    x_upper, t_upper = b_upper[..., 0:1], b_upper[..., 1:2]
    x_lower, t_lower = b_lower[..., 0:1], b_lower[..., 1:2]

    # ---- model
    model, _ = build_model(args)
    model = model.to(device)
    model.apply(init_weights)
    n_params = get_n_params(model)
    print(f"[model] {args.model}  params={n_params:,}")
    optim = LBFGS(model.parameters(), line_search_fn="strong_wolfe")

    # ---- train
    loss_track = []
    if device.startswith("cuda"):
        torch.cuda.reset_peak_memory_stats()
    t0 = time.time()
    for i in range(args.iters):
        def closure():
            r, _ = pde.residual(model, x_res, t_res)
            loss_res = torch.mean(r ** 2)
            loss_ic, loss_bc = pde.ic_bc_loss(model, x_left, t_left, x_upper, t_upper, x_lower, t_lower)
            loss = loss_res + loss_ic + loss_bc
            loss_track.append([loss_res.item(), loss_ic.item(), loss_bc.item()])
            optim.zero_grad()
            loss.backward()
            return loss
        optim.step(closure)
        if (i + 1) % 50 == 0 or i == 0:
            lr_, li_, lb_ = loss_track[-1]
            print(f"  step {i+1:5d}/{args.iters}  res={lr_:.3e} ic={li_:.3e} bc={lb_:.3e}  "
                  f"total={lr_+li_+lb_:.3e}  {time.time()-t0:6.1f}s", flush=True)
    train_time = time.time() - t0
    peak_mem_mib = torch.cuda.max_memory_allocated() / 2**20 if device.startswith("cuda") else float("nan")
    final_loss = float(np.sum(loss_track[-1]))

    # ---- evaluate (rMAE / rRMSE = relative L1 / L2, eq. 7 in the paper)
    xt_test = to_t(res_test_in)
    with torch.no_grad():
        pred = model(xt_test[..., 0:1], xt_test[..., 1:2])
        if is_seq:
            pred = pred[:, 0]  # first element of the pseudo-sequence is u(x,t)
        pred = pred.reshape(-1).cpu().numpy()
    u = pde.ground_truth(res_test).reshape(-1)
    rl1 = float(np.sum(np.abs(u - pred)) / np.sum(np.abs(u)))
    rl2 = float(np.sqrt(np.sum((u - pred) ** 2) / np.sum(u ** 2)))
    print(f"[result] loss={final_loss:.3e}  rMAE={rl1:.4f}  rRMSE={rl2:.4f}  "
          f"time={train_time:.1f}s ({train_time/args.iters:.2f}s/step)  peak_mem={peak_mem_mib:.0f}MiB")

    # ---- save
    np.save(os.path.join(run_dir, "pred.npy"), pred.reshape(args.test_grid, args.test_grid))
    with open(os.path.join(run_dir, "loss.json"), "w") as f:
        json.dump(loss_track, f)
    with open(os.path.join(run_dir, "config.json"), "w") as f:
        json.dump(vars(args), f, indent=2)
    if args.save_model:
        torch.save(model.state_dict(), os.path.join(run_dir, "model.pt"))

    try:
        import matplotlib
        matplotlib.use("Agg")
        import matplotlib.pyplot as plt
        P = pred.reshape(args.test_grid, args.test_grid)
        U = u.reshape(args.test_grid, args.test_grid)
        ext = [pde.x_range[0], pde.x_range[1], pde.t_range[1], pde.t_range[0]]
        fig, ax = plt.subplots(1, 3, figsize=(12, 3.2))
        for a, img, title in zip(ax, [U, P, np.abs(P - U)], ["Exact u(x,t)", f"{args.model} prediction", "Absolute error"]):
            im = a.imshow(img, extent=ext, aspect="auto")
            a.set_title(title); a.set_xlabel("x"); a.set_ylabel("t")
            fig.colorbar(im, ax=a)
        fig.suptitle(f"{args.pde}: rMAE={rl1:.3f} rRMSE={rl2:.3f}")
        fig.tight_layout()
        fig.savefig(os.path.join(run_dir, "figure.png"), dpi=130)
        plt.close(fig)
    except Exception as e:  # plotting is optional
        print("[warn] plotting failed:", e)

    row = dict(run=run_name, pde=args.pde, model=args.model, activation=args.activation if is_seq else
               ("wavelet" if args.model == "pinn_wavelet" else "default"), seed=args.seed, iters=args.iters,
               train_grid=args.train_grid, n_res=n_res, n_ic=n_ic, n_bc=n_bc,
               k=args.k if is_seq else 1, dt=args.dt if is_seq else 0.0,
               n_params=n_params, loss=final_loss, loss_res=loss_track[-1][0], loss_ic=loss_track[-1][1],
               loss_bc=loss_track[-1][2], rMAE=rl1, rRMSE=rl2, train_time_s=round(train_time, 1),
               s_per_step=round(train_time / args.iters, 3), peak_mem_MiB=round(peak_mem_mib, 1),
               device=device, gpu=torch.cuda.get_device_name(0) if device.startswith("cuda") else "cpu")
    csv_path = os.path.join(args.out, "results.csv")
    existing, fields = [], list(row.keys())
    if os.path.exists(csv_path):  # merge columns so older rows (without e.g. 'activation') stay valid
        with open(csv_path, newline="") as f:
            r = csv.DictReader(f)
            existing = list(r)
            fields = list(r.fieldnames or []) + [k for k in row if k not in (r.fieldnames or [])]
    with open(csv_path, "w", newline="") as f:
        w = csv.DictWriter(f, fieldnames=fields, restval="")
        w.writeheader()
        for old in existing:
            w.writerow(old)
        w.writerow(row)
    print(f"[saved] {run_dir}  ->  {csv_path}")


if __name__ == "__main__":
    main()


In [ ]:
%%writefile run_all.sh
#!/usr/bin/env bash
# Full experiment sweep for the reproducibility report.
# Usage:  bash run_all.sh [ITERS]      (default ITERS=500, as in the authors' notebooks)
# Runs are appended to results/results.csv; already-finished runs are skipped.
set -e
ITERS=${1:-500}
cd "$(dirname "$0")"

run () {  # skip a run if its directory already contains pred.npy
  local name
  name=$(python3 - "$@" <<'EOF'
import sys
a = sys.argv[1:]
def get(flag, default):
    return a[a.index(flag)+1] if flag in a else default
pde, model, seed = get('--pde', ''), get('--model', ''), get('--seed', '0')
iters = get('--iters', '500'); k = get('--k', '5')
grid = get('--train_grid', '51' if model == 'pinnsformer' else '101')
dt = get('--dt', '1e-4')
name = f"{pde}_{model}_g{grid}_it{iters}_s{seed}"
if model == 'pinnsformer':
    name += f"_k{k}_dt{float(dt):g}"
    act = get('--activation', 'wavelet')
    if act != 'wavelet': name += f"_{act}"
if '--tag' in a: name += f"_{get('--tag', '')}"
print(name)
EOF
)
  if [ -f "results/$name/pred.npy" ]; then echo "[skip] $name"; return; fi
  python3 run_experiment.py "$@"
}

echo "=================== TIER 1: Table 1 (convection, 1D-reaction), 4 models ==================="
for pde in convection 1d_reaction; do
  for model in pinn qres fls pinnsformer; do
    run --pde $pde --model $model --iters $ITERS --seed 0
  done
done

echo "=================== TIER 1b: QRes exactly as in the convection notebook (512 wide, 2 layers) ==================="
run --pde convection --model qres --iters $ITERS --seed 0 --qres_hidden 512 --qres_layers 2 --tag nbconfig

echo "=================== TIER 2a: seeds (variance the paper does not report) ==================="
for pde in convection 1d_reaction; do
  for seed in 1 2; do
    run --pde $pde --model pinn        --iters $ITERS --seed $seed
    run --pde $pde --model pinnsformer --iters $ITERS --seed $seed
  done
done

echo "=================== TIER 2b: is it the activation? PINN + Wavelet ==================="
for pde in convection 1d_reaction; do
  run --pde $pde --model pinn_wavelet --iters $ITERS --seed 0
done

echo "=================== TIER 2c: is it the pseudo-sequence? PINNsFormer k=1 ==================="
for pde in convection 1d_reaction; do
  run --pde $pde --model pinnsformer --iters $ITERS --seed 0 --k 1
done

echo "=================== TIER 2d: fairness of the training grid ==================="
for pde in convection 1d_reaction; do
  run --pde $pde --model pinn        --iters $ITERS --seed 0 --train_grid 51
  run --pde $pde --model pinnsformer --iters $ITERS --seed 0 --train_grid 101
done

echo "=================== TIER 2e: is Wavelet necessary? PINNsFormer with Sin / ReLU (paper Table 6, subset) ==================="
for pde in convection 1d_reaction; do
  run --pde $pde --model pinnsformer --iters $ITERS --seed 0 --activation sin
  run --pde $pde --model pinnsformer --iters $ITERS --seed 0 --activation relu
done

echo "=================== TIER 3: 1D-wave without NTK (Table 2, first two rows) ==================="
run --pde 1d_wave --model pinn        --iters 1000 --seed 0
run --pde 1d_wave --model pinnsformer --iters 1000 --seed 0

echo "Done. Summary:"
python3 summarize.py


In [ ]:
%%writefile summarize.py
"""Print results/results.csv as report-ready tables (and a LaTeX version).

    python summarize.py            # markdown tables to stdout
    python summarize.py --latex    # LaTeX tabular rows
"""
import argparse
import os

import pandas as pd

ROOT = os.path.dirname(os.path.abspath(__file__))
p = argparse.ArgumentParser()
p.add_argument("--csv", default=os.path.join(ROOT, "results", "results.csv"))
p.add_argument("--latex", action="store_true")
args = p.parse_args()

df = pd.read_csv(args.csv)
if "activation" not in df.columns:  # rows written before the activation column existed
    df["activation"] = "wavelet"
df["activation"] = df["activation"].fillna("wavelet")


def label(r):
    if r.model == "pinnsformer":
        s = f"pinnsformer (grid {r.train_grid}, k={r.k}"
        if r.activation not in ("wavelet", "default"):
            s += f", act={r.activation}"
        return s + ")"
    s = f"{r.model} (grid {r.train_grid})"
    if "nbconfig" in str(r.run):
        s += " [notebook config]"
    return s


df["config"] = df.apply(label, axis=1)

paper = {  # Table 1 / Table 2 of the paper, for side-by-side comparison
    ("convection", "pinn"): (0.016, 0.778, 0.840), ("convection", "qres"): (0.015, 0.746, 0.816),
    ("convection", "fls"): (0.012, 0.674, 0.771), ("convection", "pinnsformer"): (3.7e-5, 0.023, 0.027),
    ("1d_reaction", "pinn"): (0.199, 0.982, 0.981), ("1d_reaction", "qres"): (0.199, 0.979, 0.977),
    ("1d_reaction", "fls"): (0.2, 0.984, 0.985), ("1d_reaction", "pinnsformer"): (3.0e-6, 0.015, 0.030),
    ("1d_wave", "pinn"): (1.93e-2, 0.326, 0.335), ("1d_wave", "pinnsformer"): (1.38e-2, 0.270, 0.283),
}

for pde, g in df.groupby("pde", sort=False):
    print(f"\n## {pde}\n")
    agg = g.groupby("config").agg(seeds=("seed", "count"), loss=("loss", "mean"),
                                  rMAE=("rMAE", "mean"), rMAE_std=("rMAE", "std"),
                                  rRMSE=("rRMSE", "mean"), rRMSE_std=("rRMSE", "std"),
                                  s_per_step=("s_per_step", "mean"), mem_MiB=("peak_mem_MiB", "mean"),
                                  params=("n_params", "first"), model=("model", "first"),
                                  grid=("train_grid", "first"), k=("k", "first"), act=("activation", "first"))
    rows = []
    for cfg, r in agg.iterrows():
        default_grid = (r.grid == 51) if r.model == "pinnsformer" else (r.grid == 101)
        std_pf = r.model != "pinnsformer" or (r.k == 5 and r.act in ("wavelet", "default"))
        ref = paper.get((pde, r.model)) if (default_grid and std_pf and "nbconfig" not in cfg) else None
        if r.model == "pinnsformer" and r.act in ("sin", "relu") and default_grid:  # paper Table 6
            ref = {("convection", "sin"): (0.3159, 1.074, 1.141), ("convection", "relu"): (0.5256, 1.001, 1.001),
                   ("1d_reaction", "sin"): (4.9e-6, 0.017, 0.032), ("1d_reaction", "relu"): (0.2083, 0.994, 0.996)}.get((pde, r.act))
        rows.append(dict(config=cfg, seeds=int(r.seeds), params=int(r.params), loss=f"{r.loss:.2e}",
                         rMAE=f"{r.rMAE:.3f}" + (f" ± {r.rMAE_std:.3f}" if r.seeds > 1 else ""),
                         rRMSE=f"{r.rRMSE:.3f}" + (f" ± {r.rRMSE_std:.3f}" if r.seeds > 1 else ""),
                         paper_rMAE=f"{ref[1]:.3f}" if ref else "-", paper_rRMSE=f"{ref[2]:.3f}" if ref else "-",
                         s_per_step=f"{r.s_per_step:.2f}", mem_MiB=f"{r.mem_MiB:.0f}"))
    out = pd.DataFrame(rows)
    if args.latex:
        for _, r in out.iterrows():
            print(f"{r.config} & {r.loss} & {r.rMAE} & {r.rRMSE} & {r.paper_rMAE} & {r.paper_rRMSE} & {r.s_per_step} & {r.mem_MiB} \\\\")
    else:
        print(out.to_string(index=False))


## (Optional) mount Google Drive so results survive a disconnect
Skip if you prefer to download `results.zip` manually at the end.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!mkdir -p /content/drive/MyDrive/pinnsformer_results
# If you have results from a previous session, restore them so finished runs are skipped:
!cp -rn /content/drive/MyDrive/pinnsformer_results/results . 2>/dev/null || true

## Sanity check (about 1 minute)
A short run to make sure everything works before the long sweep.

In [ ]:
!python run_experiment.py --pde convection --model pinnsformer --iters 5 --train_grid 21 --out results_smoke
!rm -rf results_smoke

## Option A: run the whole study in one cell
This runs Tier 1, 2 and 3 in priority order (about 5-7 GPU hours on a T4), copies results to Drive after every run, and skips runs that already exist. If Colab disconnects, just re-run the setup cells above and this cell again; it continues where it stopped. Skip to Option B if you prefer to run tier by tier.

In [ ]:
import subprocess, sys
# stream output live and back up after each run
proc = subprocess.Popen(['bash', 'run_all.sh', '500'], stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
for line in proc.stdout:
    print(line, end='')
    if line.startswith('[saved]'):
        subprocess.run('cp -r results /content/drive/MyDrive/pinnsformer_results/ 2>/dev/null', shell=True)
proc.wait()

## Option B, Tier 1: Table 1 (convection + 1D-reaction, four models)
About 8 runs. Expect roughly 5-10 min per MLP baseline and 20-40 min per PINNsFormer run on a T4. `ITERS=500` matches the authors' notebooks; the paper text says 1000. You can run the whole sweep with `bash run_all.sh` instead (it also does Tier 2 and 3 and skips finished runs).

In [ ]:
ITERS = 500
for pde in ['convection', '1d_reaction']:
    for model in ['pinn', 'qres', 'fls', 'pinnsformer']:
        !python run_experiment.py --pde {pde} --model {model} --iters {ITERS} --seed 0
        !cp -r results /content/drive/MyDrive/pinnsformer_results/ 2>/dev/null || true
# QRes with the configuration the convection notebook actually uses (512 wide, 2 layers; the paper says 256 x 4)
!python run_experiment.py --pde convection --model qres --iters {ITERS} --seed 0 --qres_hidden 512 --qres_layers 2 --tag nbconfig

In [ ]:
!python summarize.py

## Option B, Tier 2: beyond the paper
* **Seeds**: the paper reports single runs. Two more seeds for PINN and PINNsFormer.
* **PINN + Wavelet**: is the gain from the Transformer or from the activation?
* **PINNsFormer with k=1**: removes the pseudo-sequence, keeps the architecture.
* **Grid fairness**: the paper trains baselines on 101x101 but PINNsFormer on 51x51.

In [ ]:
for pde in ['convection', '1d_reaction']:
    for seed in [1, 2]:
        !python run_experiment.py --pde {pde} --model pinn        --iters {ITERS} --seed {seed}
        !python run_experiment.py --pde {pde} --model pinnsformer --iters {ITERS} --seed {seed}
    !python run_experiment.py --pde {pde} --model pinn_wavelet --iters {ITERS} --seed 0
    !python run_experiment.py --pde {pde} --model pinnsformer  --iters {ITERS} --seed 0 --k 1
    !python run_experiment.py --pde {pde} --model pinn         --iters {ITERS} --seed 0 --train_grid 51
    !python run_experiment.py --pde {pde} --model pinnsformer  --iters {ITERS} --seed 0 --train_grid 101
    !cp -r results /content/drive/MyDrive/pinnsformer_results/ 2>/dev/null || true

In [ ]:
!python summarize.py

## Option B, Tier 2e: is Wavelet necessary? (subset of the paper's Table 6)
PINNsFormer with every Wavelet activation replaced by Sin or ReLU. The paper reports both fall back into the failure regime.

In [ ]:
for pde in ['convection', '1d_reaction']:
    for act in ['sin', 'relu']:
        !python run_experiment.py --pde {pde} --model pinnsformer --iters {ITERS} --seed 0 --activation {act}
    !cp -r results /content/drive/MyDrive/pinnsformer_results/ 2>/dev/null || true

## Option B, Tier 3 (if time): 1D-wave without NTK (first two rows of Table 2)
The NTK variants are in the authors' notebooks `demo/1d_wave/*_ntk.ipynb`; they need 1000 iterations.

In [ ]:
!python run_experiment.py --pde 1d_wave --model pinn        --iters 1000 --seed 0
!python run_experiment.py --pde 1d_wave --model pinnsformer --iters 1000 --seed 0
!cp -r results /content/drive/MyDrive/pinnsformer_results/ 2>/dev/null || true

## Demo: the authors' pretrained checkpoint (no training needed)
Useful for the live demo in the presentation. Loads `checkpoint/convection_pinnsformer.pt` and plots it against the ground truth.

In [ ]:
import numpy as np, torch, scipy.io, matplotlib.pyplot as plt
from util import get_data, make_time_sequence
from model.pinnsformer import PINNsformer
dev = 'cuda' if torch.cuda.is_available() else 'cpu'
model = PINNsformer(d_out=1, d_hidden=512, d_model=32, N=1, heads=2).to(dev)
model.load_state_dict(torch.load('checkpoint/convection_pinnsformer.pt', map_location=dev))
model.eval()
res_test, *_ = get_data([0, 2*np.pi], [0, 1], 101, 101)
xt = torch.tensor(make_time_sequence(res_test, 5, 1e-4), dtype=torch.float32).to(dev)
with torch.no_grad():
    pred = model(xt[..., 0:1], xt[..., 1:2])[:, 0].cpu().numpy().reshape(101, 101)
u = scipy.io.loadmat('demo/convection/convection.mat')['u'].reshape(101, 101)
print('rMAE', np.abs(u-pred).sum()/np.abs(u).sum(), ' rRMSE', np.sqrt(((u-pred)**2).sum()/(u**2).sum()))
fig, ax = plt.subplots(1, 3, figsize=(12, 3.2))
for a, img, t in zip(ax, [u, pred, np.abs(u-pred)], ['Exact', 'Pretrained PINNsFormer', 'Abs. error']):
    im = a.imshow(img, extent=[0, 2*np.pi, 1, 0], aspect='auto'); a.set_title(t); fig.colorbar(im, ax=a)
plt.tight_layout(); plt.show()

## Collect everything
Download `results.zip` (CSV, per-run predictions, loss curves, figures) for the report.

In [ ]:
!python summarize.py --latex
!zip -qr results.zip results
!cp results.zip /content/drive/MyDrive/pinnsformer_results/ 2>/dev/null || true
from google.colab import files; files.download('results.zip')